# 01 - Data cleaning and preprocessing

**Goal:** turn the raw file `data/raw/customer_churn_raw_1050.csv` into clean train / validation / test files,
and keep a log of every change we made and why.

The raw file is synthetic (1,050 rows x 33 columns) and contains deliberate data-quality problems:
duplicate rows, a duplicate column, negative values that cannot be negative, out-of-range CSAT scores,
inconsistent category labels and missing values.

**Order of work (and why):**
1. Fix problems that are *errors* (duplicates, impossible values, typos in labels). These fixes do not
   depend on any statistics, so they can be applied to the whole file.
2. Split the data **by time** into train / validation / test.
3. Only then fill missing values, using fill values learned from the **train** split only.
   This way the validation and test months never influence the training data (no leakage).

Outputs:
- `data/processed/train_clean.csv`, `validation_clean.csv`, `test_clean.csv`
- `reports/cleaning_log.csv`

The modelling files `data/*_final.csv` are **not** touched.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Work from the repository root whether the notebook is started from notebooks/ or the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_FILE = ROOT / "data" / "raw" / "customer_churn_raw_1050.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
REPORTS_DIR = ROOT / "reports"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Every cleaning step adds one row here; it becomes the cleaning-log table at the end
cleaning_log = []
def log_step(step, affected, action, reason):
    cleaning_log.append({"step": step, "rows_or_values_affected": affected, "action": action, "reason": reason})

## 1. Load and inspect

First we look at the data before changing anything: size, column types, a few rows,
summary statistics, missing values and the distinct values of the text columns.

In [2]:
raw = pd.read_csv(RAW_FILE)
print("Shape:", raw.shape)
raw.head()

Shape: (1050, 33)


,customer_id,snapshot_date,signup_date,plan_type,contract_type,contract_end_date,acquisition_channel,region,payment_method,paperless_billing,monthly_charge_current,monthly_charge_3m_ago,bill_amount,discount_amount,num_addon_services,usage_hours_m1,usage_hours_m2,usage_hours_m3,app_logins_last_30d,last_login_date,support_tickets_m1,support_tickets_m2,support_tickets_m3,tickets_resolved_last_90d,complaints_last_90d,last_csat_score,last_bill_due_date,last_payment_date,late_payments_last_6m,billing_disputes_last_6m,total_billed_last_6m,total_paid_last_6m,churn_next_30d
0,CUST200524,2025-06-30,2024-01-09,Basic,One year,2026-01-08,Online,East,Credit card,Yes,272.75,272.75,272.75,0.0,3,21.6,20.8,20.4,12.0,2025-06-26,0,1,0,1,0,3.0,2025-06-13,2025-06-08,1,0,1636.50,1636.50,0
1,CUST200254,2025-03-31,2024-04-19,Basic,Month-to-month,NaN,Call center,South,Autopay,No,271.07,271.07,271.07,50.0,3,19.6,24.4,30.0,5.0,2025-02-20,0,1,3,2,1,NaN,2025-03-07,2025-03-05,0,0,1326.42,1326.42,1
2,CUST200832,2025-12-31,2024-01-18,Family,Month-to-month,NaN,Online,South,Cash,No,1094.55,1094.55,1094.55,50.0,0,8.9,17.1,24.9,5.0,2025-12-27,2,1,0,0,0,4.0,2025-12-20,2025-12-15,1,0,6267.30,6267.30,1
3,CUST200343,2025-11-30,2025-07-27,Basic,Month-to-month,NaN,Online,South,Credit card,No,300.10,300.10,300.10,0.0,1,40.2,39.6,37.8,11.0,2025-11-19,0,0,0,0,0,4.0,2025-11-19,2025-11-18,1,0,1800.60,1800.60,0
4,CUST200601,2025-12-31,2019-11-22,Standard,Month-to-month,NaN,Online,North,UPI,No,510.24,510.24,510.24,0.0,1,57.0,54.7,50.0,9.0,2025-12-29,1,0,0,1,0,5.0,2025-12-11,2025-12-10,1,0,3061.44,3061.44,0


In [3]:
raw.dtypes.to_frame("dtype").T

,customer_id,snapshot_date,signup_date,plan_type,contract_type,contract_end_date,acquisition_channel,region,payment_method,paperless_billing,monthly_charge_current,monthly_charge_3m_ago,bill_amount,discount_amount,num_addon_services,usage_hours_m1,usage_hours_m2,usage_hours_m3,app_logins_last_30d,last_login_date,support_tickets_m1,support_tickets_m2,support_tickets_m3,tickets_resolved_last_90d,complaints_last_90d,last_csat_score,last_bill_due_date,last_payment_date,late_payments_last_6m,billing_disputes_last_6m,total_billed_last_6m,total_paid_last_6m,churn_next_30d
dtype,str,str,str,str,str,str,str,str,str,str,float64,float64,float64,float64,int64,float64,float64,float64,float64,str,int64,int64,int64,int64,int64,float64,str,str,int64,int64,float64,float64,int64


In [4]:
raw.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
monthly_charge_current,1050.0,606.55,337.39,-1230.20,323.35,559.20,859.49,1491.44
monthly_charge_3m_ago,1050.0,592.48,296.44,269.23,311.78,539.99,847.07,1309.64
bill_amount,1050.0,619.18,313.59,269.23,325.00,561.62,864.14,1491.44
discount_amount,1050.0,47.52,57.67,-150.00,0.00,0.00,100.00,150.00
num_addon_services,1050.0,1.54,1.12,0.00,1.00,2.00,3.00,3.00
usage_hours_m1,1050.0,35.30,17.45,-47.80,21.90,34.20,47.58,90.40
usage_hours_m2,1019.0,38.00,14.84,0.00,27.40,37.70,48.45,87.10
usage_hours_m3,1030.0,40.32,14.44,0.00,30.10,40.80,50.62,88.10
app_logins_last_30d,1035.0,10.13,4.24,1.00,7.00,10.00,13.00,23.00
support_tickets_m1,1050.0,0.40,0.67,0.00,0.00,0.00,1.00,4.00


Already visible in the summary: `monthly_charge_current`, `usage_hours_m1` and `discount_amount` have
**negative minimums**, which is impossible for a charge, hours of use or a discount.
`last_csat_score` has a max of 9 and a min of 0, but CSAT is a 1-5 scale.

In [5]:
missing = raw.isna().sum()
missing = missing[missing > 0].to_frame("missing_values")
missing["missing_pct"] = (missing["missing_values"] / len(raw) * 100).round(1)
missing

,missing_values,missing_pct
contract_end_date,581,55.3
acquisition_channel,20,1.9
region,12,1.1
usage_hours_m2,31,3.0
usage_hours_m3,20,1.9
app_logins_last_30d,15,1.4
last_csat_score,48,4.6
last_payment_date,24,2.3


In [6]:
text_cols = ["plan_type", "contract_type", "acquisition_channel", "region", "payment_method", "paperless_billing"]
for col in text_cols:
    print(col, raw[col].value_counts(dropna=False).to_dict())

plan_type {'Standard': 380, 'Basic': 371, 'Premium': 189, 'Family': 110}
contract_type {'Month-to-month': 570, 'One year': 271, 'Two year': 189, 'Month to Month': 8, 'MTM': 6, 'month-to-month': 6}
acquisition_channel {'Online': 448, 'Retail store': 330, 'Call center': 146, 'Partner': 106, nan: 20}
region {'South': 296, 'North': 255, 'East': 248, 'West': 239, nan: 12}
payment_method {'Autopay': 384, 'UPI': 310, 'Credit card': 227, 'Cash': 119, 'AUTOPAY': 6, 'autopay': 4}
paperless_billing {'Yes': 597, 'No': 453}


`contract_type` has four spellings of the same thing (`Month-to-month`, `Month to Month`, `MTM`,
`month-to-month`) and `payment_method` has three spellings of `Autopay`. A model would treat these as
different categories, so they must be unified.

## 2. Duplicate rows

An exact duplicate row is the same customer snapshot recorded twice. Keeping it would give that customer
double weight in training and could put the same row in two places. We remove exact duplicates and then
check that each `customer_id` + `snapshot_date` pair is unique.

In [7]:
n_dup = raw.duplicated().sum()
print("Exact duplicate rows:", n_dup)

df = raw.drop_duplicates().reset_index(drop=True)
print("Rows after removing duplicates:", len(df))
print("Duplicate customer_id + snapshot_date pairs:", df.duplicated(["customer_id", "snapshot_date"]).sum())
print("Duplicate customer_id (any date):", df["customer_id"].duplicated().sum())

log_step("Duplicate rows", int(n_dup), "Dropped exact duplicate rows (1,050 -> %d)" % len(df),
         "Same snapshot recorded twice; would double-count customers")

Exact duplicate rows: 30
Rows after removing duplicates: 1020
Duplicate customer_id + snapshot_date pairs: 0
Duplicate customer_id (any date): 0


Each customer now appears exactly once, so the same customer can never end up in both train and test.

## 3. Duplicate column: `bill_amount` vs `monthly_charge_current`

`bill_amount` looks like a copy of `monthly_charge_current`. If it is a copy, it adds no information and
only creates a perfectly correlated pair of columns. We check it, use it to fill any missing current
charge, then drop it.

In [8]:
both = df["bill_amount"].notna() & df["monthly_charge_current"].notna()
exact_equal = np.isclose(df.loc[both, "bill_amount"], df.loc[both, "monthly_charge_current"])
equal_ignoring_sign = np.isclose(df.loc[both, "bill_amount"], df.loc[both, "monthly_charge_current"].abs())

print("Rows where both exist:", both.sum())
print("Exactly equal:", exact_equal.sum())
print("Equal after ignoring the sign:", equal_ignoring_sign.sum())
df.loc[both & ~np.isclose(df["bill_amount"], df["monthly_charge_current"]),
       ["customer_id", "monthly_charge_current", "bill_amount"]]

Rows where both exist: 1020
Exactly equal: 1012
Equal after ignoring the sign: 1020


,customer_id,monthly_charge_current,bill_amount
8,CUST200060,-915.99,915.99
383,CUST200680,-551.72,551.72
575,CUST200090,-295.61,295.61
708,CUST200119,-360.93,360.93
767,CUST200487,-1230.20,1230.20
841,CUST200708,-548.48,548.48
876,CUST200455,-869.72,869.72
1006,CUST200041,-942.25,942.25


`bill_amount` equals `monthly_charge_current` on every row once the sign is ignored. The only rows that
differ are the ones where `monthly_charge_current` was stored as a **negative** number (fixed in step 4).
This proves the column is a duplicate. `monthly_charge_current` has no missing values in this file, so
the fill step changes 0 rows, but we keep it in the code so the rule is documented.

In [9]:
n_charge_missing = df["monthly_charge_current"].isna().sum()
df["monthly_charge_current"] = df["monthly_charge_current"].fillna(df["bill_amount"])
df = df.drop(columns="bill_amount")

print("Missing monthly_charge_current filled from bill_amount:", n_charge_missing)
print("Columns now:", df.shape[1])

log_step("Duplicate column", int(n_charge_missing),
         "Filled missing monthly_charge_current from bill_amount, then dropped bill_amount",
         "bill_amount equals monthly_charge_current on all %d rows (ignoring sign)" % both.sum())

Missing monthly_charge_current filled from bill_amount: 0
Columns now: 32


## 4. Invalid negative values

A monthly charge, hours of usage and a discount can never be negative. The negative numbers have the
right size but the wrong sign (step 3 showed `bill_amount` holds the same charge as a positive number),
so for charge and usage we keep the size and drop the sign (absolute value).
A negative discount does not make sense, so it is set to 0 (no discount).

**Important:** later we will compute *usage change* (this month minus three months ago). That change
**can** be negative: it simply means usage went down, which is one of the strongest churn signals.
Those negative values are valid and must not be "fixed".

In [10]:
abs_cols = ["monthly_charge_current", "usage_hours_m1", "usage_hours_m2", "usage_hours_m3"]
neg_before = {col: int((df[col] < 0).sum()) for col in abs_cols + ["discount_amount"]}

for col in abs_cols:
    df[col] = df[col].abs()              # NaN stays NaN
df.loc[df["discount_amount"] < 0, "discount_amount"] = 0

neg_after = {col: int((df[col] < 0).sum()) for col in abs_cols + ["discount_amount"]}
pd.DataFrame({"negative_before": neg_before, "negative_after": neg_after})

,negative_before,negative_after
monthly_charge_current,8,0
usage_hours_m1,5,0
usage_hours_m2,0,0
usage_hours_m3,0,0
discount_amount,5,0


In [11]:
for col in abs_cols:
    if neg_before[col] > 0:
        log_step("Negative values", neg_before[col], f"{col}: replaced with absolute value",
                 "A charge / usage hours cannot be negative; sign error")
log_step("Negative values", neg_before["discount_amount"], "discount_amount: negative set to 0",
         "A negative discount is invalid; treated as no discount")

## 5. Out-of-range CSAT scores

`last_csat_score` is a 1-5 survey score. Values such as 0, 7 or 9 cannot be real answers, so we treat
them as unknown (`NaN`). They are filled together with the other missing CSAT values in step 9.

In [12]:
out_of_range = df["last_csat_score"].notna() & ~df["last_csat_score"].between(1, 5)
print("Out-of-range CSAT values:", df.loc[out_of_range, "last_csat_score"].tolist())
df.loc[out_of_range, "last_csat_score"] = np.nan
print("CSAT missing after this step:", df["last_csat_score"].isna().sum())

log_step("Out-of-range values", int(out_of_range.sum()), "last_csat_score outside 1-5 set to NaN",
         "CSAT is a 1-5 scale; 0/7/9 are recording errors")

Out-of-range CSAT values: [9.0, 7.0, 0.0, 0.0]
CSAT missing after this step: 49


## 6. Inconsistent category labels

Same meaning, different spelling. We map every variant to one standard label.

In [13]:
contract_map = {"MTM": "Month-to-month", "Month to Month": "Month-to-month", "month-to-month": "Month-to-month"}
payment_map = {"autopay": "Autopay", "AUTOPAY": "Autopay"}

print("BEFORE")
print(df["contract_type"].value_counts().to_dict())
print(df["payment_method"].value_counts().to_dict())

n_contract = int(df["contract_type"].isin(contract_map).sum())
n_payment = int(df["payment_method"].isin(payment_map).sum())
df["contract_type"] = df["contract_type"].replace(contract_map)
df["payment_method"] = df["payment_method"].replace(payment_map)

print("\nAFTER")
print(df["contract_type"].value_counts().to_dict())
print(df["payment_method"].value_counts().to_dict())

log_step("Inconsistent labels", n_contract, "contract_type variants -> 'Month-to-month'",
         "Same category spelled differently would become separate categories")
log_step("Inconsistent labels", n_payment, "payment_method variants -> 'Autopay'",
         "Same category spelled differently would become separate categories")

BEFORE
{'Month-to-month': 554, 'One year': 262, 'Two year': 184, 'Month to Month': 8, 'MTM': 6, 'month-to-month': 6}
{'Autopay': 371, 'UPI': 305, 'Credit card': 219, 'Cash': 115, 'AUTOPAY': 6, 'autopay': 4}

AFTER
{'Month-to-month': 574, 'One year': 262, 'Two year': 184}
{'Autopay': 381, 'UPI': 305, 'Credit card': 219, 'Cash': 115}


## 7. Convert dates

Dates are stored as text. Converting them to real dates lets us subtract them later
(for example *days since last login*).

In [14]:
date_cols = ["snapshot_date", "signup_date", "contract_end_date", "last_login_date",
             "last_bill_due_date", "last_payment_date"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])
df[date_cols].dtypes

snapshot_date         datetime64[us]
signup_date           datetime64[us]
contract_end_date     datetime64[us]
last_login_date       datetime64[us]
last_bill_due_date    datetime64[us]
last_payment_date     datetime64[us]
dtype: object

In [15]:
# Sanity check: no event date should be after the snapshot date (that would be future information)
for col in ["signup_date", "last_login_date", "last_bill_due_date", "last_payment_date"]:
    print(f"{col} later than snapshot_date:", (df[col] > df["snapshot_date"]).sum())

signup_date later than snapshot_date: 0
last_login_date later than snapshot_date: 0
last_bill_due_date later than snapshot_date: 0
last_payment_date later than snapshot_date: 0


## 8. Chronological split (before filling missing values)

The model will be used to score customers **in the future** using what we learned from the past.
So we test it the same way:

| Split | Snapshot months | Used for |
|---|---|---|
| Train | Jan - Aug 2025 | learning the model and all fill values |
| Validation | Sep - Oct 2025 | tuning and model choice |
| Test | Nov - Dec 2025 | one final, honest check |

A random split would mix December rows into training and let the model "see the future".
We split **before** imputation so that medians and other fill values come only from the training months.

In [16]:
month = df["snapshot_date"].dt.month
split_name = np.select([month <= 8, month <= 10], ["train", "validation"], default="test")
splits = {name: df[split_name == name].copy() for name in ["train", "validation", "test"]}

summary = pd.DataFrame({
    name: {"rows": len(s), "first_snapshot": s["snapshot_date"].min().date(),
           "last_snapshot": s["snapshot_date"].max().date(),
           "churners": int(s["churn_next_30d"].sum()), "churn_rate": round(s["churn_next_30d"].mean(), 3)}
    for name, s in splits.items()
}).T
summary

,rows,first_snapshot,last_snapshot,churners,churn_rate
train,665,2025-01-31,2025-08-31,151,0.227
validation,172,2025-09-30,2025-10-31,43,0.25
test,183,2025-11-30,2025-12-31,32,0.175


## 9. Missing values (fill values learned from TRAIN only)

| Column | Rule | Why |
|---|---|---|
| `last_csat_score` | add `csat_missing` flag, then train median | keeps the fact that the score was unknown |
| `usage_hours_m2` | mean of m1 and m3 | the middle month is best estimated from its neighbours |
| `usage_hours_m1` | m2, then m3 | nearest available month |
| `usage_hours_m3` | m2 | nearest month |
| `app_logins_last_30d` | train median | typical value; robust to outliers |
| `discount_amount` | 0 | no discount recorded = no discount |
| `acquisition_channel`, `region` | `"Unknown"` | do not guess a category; keep it as its own group |

**Valid blanks (do NOT fill):**
- `contract_end_date` is blank for month-to-month customers: they have no end date. We check this below.
- `last_payment_date` is blank when the last bill has **not been paid yet**. That is real information, so
  we create `bill_unpaid_flag` instead of inventing a date.

In [17]:
train = splits["train"]
fill_values = {
    "last_csat_score": train["last_csat_score"].median(),
    "app_logins_last_30d": train["app_logins_last_30d"].median(),
}
print("Fill values learned from train:", fill_values)

# Missing counts before filling, per split
fill_cols = ["last_csat_score", "usage_hours_m1", "usage_hours_m2", "usage_hours_m3", "app_logins_last_30d",
             "discount_amount", "acquisition_channel", "region", "contract_end_date", "last_payment_date"]
missing_before = pd.DataFrame({name: s[fill_cols].isna().sum() for name, s in splits.items()})
missing_before["total"] = missing_before.sum(axis=1)
missing_before

Fill values learned from train: {'last_csat_score': np.float64(4.0), 'app_logins_last_30d': np.float64(10.0)}


,train,validation,test,total
last_csat_score,33,8,8,49
usage_hours_m1,0,0,0,0
usage_hours_m2,17,9,4,30
usage_hours_m3,12,3,5,20
app_logins_last_30d,10,3,2,15
discount_amount,0,0,0,0
acquisition_channel,10,2,6,18
region,7,3,2,12
contract_end_date,365,101,99,565
last_payment_date,17,4,3,24


In [18]:
# Check that blank contract_end_date really means month-to-month
blank_end = df["contract_end_date"].isna()
pd.crosstab(df["contract_type"], blank_end.rename("contract_end_date is blank"))

contract_end_date is blank,False,True
contract_type,,
Month-to-month,9,565
One year,262,0
Two year,184,0


Every blank `contract_end_date` belongs to a month-to-month customer, which confirms the blank is valid and
not missing data. One-year and two-year contracts always have an end date.

**Data-quality note:** a small number of month-to-month customers *do* have an end date (see below).
All of them are rows whose label was one of the messy spellings fixed in step 6, and their end dates
look like fixed-term contracts, so the original label may have been wrong. We cannot know the true
contract, so we keep them as month-to-month (the same as in `data/*_final.csv`), keep their end date, and
record them in the cleaning log.

In [19]:
mtm_with_end = df["contract_type"].eq("Month-to-month") & df["contract_end_date"].notna()
mtm_with_end_ids = df.loc[mtm_with_end, "customer_id"]
print("Month-to-month customers with an end date:", mtm_with_end.sum())
print("...of which had a messy contract label in the raw file:",
      raw.drop_duplicates().loc[lambda r: r["customer_id"].isin(mtm_with_end_ids), "contract_type"]
         .isin(contract_map).sum())
log_step("Data-quality note", int(mtm_with_end.sum()), "Documented only, not changed",
         "Month-to-month (messy label) but has a contract end date; true contract unknown")

Month-to-month customers with an end date: 9
...of which had a messy contract label in the raw file: 9


In [20]:
def apply_fills(data, fills):
    """Fill missing values with the rules above. `fills` holds values learned from train only."""
    data = data.copy()
    data["csat_missing"] = data["last_csat_score"].isna().astype(int)
    data["last_csat_score"] = data["last_csat_score"].fillna(fills["last_csat_score"])

    # usage: middle month first (average of neighbours), then edges from the nearest month
    data["usage_hours_m2"] = data["usage_hours_m2"].fillna(data[["usage_hours_m1", "usage_hours_m3"]].mean(axis=1))
    data["usage_hours_m1"] = data["usage_hours_m1"].fillna(data["usage_hours_m2"]).fillna(data["usage_hours_m3"])
    data["usage_hours_m3"] = data["usage_hours_m3"].fillna(data["usage_hours_m2"])

    data["app_logins_last_30d"] = data["app_logins_last_30d"].fillna(fills["app_logins_last_30d"])
    data["discount_amount"] = data["discount_amount"].fillna(0)
    data["acquisition_channel"] = data["acquisition_channel"].fillna("Unknown")
    data["region"] = data["region"].fillna("Unknown")

    # valid blank: no payment date = bill not paid yet
    data["bill_unpaid_flag"] = data["last_payment_date"].isna().astype(int)
    return data

clean = {name: apply_fills(s, fill_values) for name, s in splits.items()}

In [21]:
rules = {
    "last_csat_score": "csat_missing flag + train median (%.1f)" % fill_values["last_csat_score"],
    "usage_hours_m1": "filled from m2, then m3",
    "usage_hours_m2": "mean of m1 and m3",
    "usage_hours_m3": "filled from m2",
    "app_logins_last_30d": "train median (%.0f)" % fill_values["app_logins_last_30d"],
    "discount_amount": "0",
    "acquisition_channel": "'Unknown'",
    "region": "'Unknown'",
}
for col, rule in rules.items():
    log_step("Missing values", int(missing_before.loc[col, "total"]), f"{col}: {rule}",
             "Fill values learned on train only, applied to all splits")
log_step("Valid blanks", int(missing_before.loc["contract_end_date", "total"]),
         "contract_end_date left blank", "Month-to-month customers have no contract end date")
log_step("Valid blanks", int(missing_before.loc["last_payment_date", "total"]),
         "last_payment_date left blank; bill_unpaid_flag created", "Blank means the last bill is unpaid")

missing_after = pd.DataFrame({name: s[fill_cols].isna().sum() for name, s in clean.items()})
missing_after

,train,validation,test
last_csat_score,0,0,0
usage_hours_m1,0,0,0
usage_hours_m2,0,0,0
usage_hours_m3,0,0,0
app_logins_last_30d,0,0,0
discount_amount,0,0,0
acquisition_channel,0,0,0
region,0,0,0
contract_end_date,365,101,99
last_payment_date,17,4,3


Only the two valid blanks remain (`contract_end_date`, `last_payment_date`).

## 10. Final checks

We assert the rules instead of only printing them, so the notebook stops with an error if anything
is wrong.

In [22]:
all_clean = pd.concat(clean.values())
allowed_blanks = {"contract_end_date", "last_payment_date"}

checks = {
    "rows (expect 1,020)": len(all_clean),
    "duplicate rows": int(all_clean.duplicated().sum()),
    "duplicate customer_id + snapshot_date": int(all_clean.duplicated(["customer_id", "snapshot_date"]).sum()),
    "negative charge / usage / discount": int((all_clean[abs_cols + ["discount_amount"]] < 0).sum().sum()),
    "CSAT outside 1-5": int((~all_clean["last_csat_score"].between(1, 5)).sum()),
    "unexpected nulls": int(all_clean.drop(columns=list(allowed_blanks)).isna().sum().sum()),
    "contract_type values": sorted(all_clean["contract_type"].unique()),
    "payment_method values": sorted(all_clean["payment_method"].unique()),
}
for name, value in checks.items():
    print(f"{name:40s} {value}")

assert len(all_clean) == 1020
assert checks["duplicate rows"] == 0 and checks["duplicate customer_id + snapshot_date"] == 0
assert checks["negative charge / usage / discount"] == 0
assert checks["CSAT outside 1-5"] == 0
assert checks["unexpected nulls"] == 0
assert checks["contract_type values"] == ["Month-to-month", "One year", "Two year"]
assert checks["payment_method values"] == ["Autopay", "Cash", "Credit card", "UPI"]
print("\nAll checks passed.")

rows (expect 1,020)                      1020
duplicate rows                           0
duplicate customer_id + snapshot_date    0
negative charge / usage / discount       0
CSAT outside 1-5                         0
unexpected nulls                         0
contract_type values                     ['Month-to-month', 'One year', 'Two year']
payment_method values                    ['Autopay', 'Cash', 'Credit card', 'UPI']

All checks passed.


## 11. Cleaning log

In [23]:
log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(REPORTS_DIR / "cleaning_log.csv", index=False)
log_df

,step,rows_or_values_affected,action,reason
0,Duplicate rows,30,"Dropped exact duplicate rows (1,050 -> 1020)",Same snapshot recorded twice; would double-cou...
1,Duplicate column,0,Filled missing monthly_charge_current from bil...,bill_amount equals monthly_charge_current on a...
2,Negative values,8,monthly_charge_current: replaced with absolute...,A charge / usage hours cannot be negative; sig...
3,Negative values,5,usage_hours_m1: replaced with absolute value,A charge / usage hours cannot be negative; sig...
4,Negative values,5,discount_amount: negative set to 0,A negative discount is invalid; treated as no ...
5,Out-of-range values,4,last_csat_score outside 1-5 set to NaN,CSAT is a 1-5 scale; 0/7/9 are recording errors
6,Inconsistent labels,20,contract_type variants -> 'Month-to-month',Same category spelled differently would become...
7,Inconsistent labels,10,payment_method variants -> 'Autopay',Same category spelled differently would become...
8,Data-quality note,9,"Documented only, not changed",Month-to-month (messy label) but has a contrac...
9,Missing values,49,last_csat_score: csat_missing flag + train med...,"Fill values learned on train only, applied to ..."


## 12. Save the cleaned splits

Saved to `data/processed/`. The modelling files `data/*_final.csv` are left unchanged;
notebook 03 rebuilds them from these clean files and compares the result.

In [24]:
for name, data in clean.items():
    out = data.sort_values(["snapshot_date", "customer_id"]).copy()
    for col in date_cols:
        out[col] = out[col].dt.strftime("%Y-%m-%d")
    path = PROCESSED_DIR / f"{name}_clean.csv"
    out.to_csv(path, index=False)
    print(f"{path.relative_to(ROOT)}: {out.shape[0]} rows x {out.shape[1]} columns")

data\processed\train_clean.csv: 665 rows x 34 columns


data\processed\validation_clean.csv: 172 rows x 34 columns
data\processed\test_clean.csv: 183 rows x 34 columns
